# Sesión 7 · Taller: tablero de resultados de un clasificador
**Misión 9–11 nov 2026 · Miércoles 11 · segunda mitad (~45 min)**

**Encargo:** una empresa quiere leer automáticamente dígitos escritos a mano (`digits`, del 0 al 9). Se probaron 3 modelos y ya tenemos sus predicciones en un CSV. Debes construir un tablero que responda: **¿qué modelo usar y en qué dígitos se equivoca más?**

Completa las celdas `# TODO` (reemplaza `...`).

| Paso | Tema | Min |
|---|---|---|
| 0 | Generar el CSV de predicciones (ya hecho, solo ejecutar) | 2 |
| 1 | Cargar el CSV | 3 |
| 2 | Accuracy de cada modelo | 7 |
| 3 | Matriz de confusión | 7 |
| 4 | % de aciertos por dígito | 7 |
| 5 | Layout + callback | 12 |
| 6 | Conclusiones | 7 |

In [ ]:
# En Google Colab: instalar lo que no viene por defecto (en local puede omitirse)
import importlib.util, subprocess, sys
for pkg in ["dash", "plotly"]:
    if importlib.util.find_spec(pkg.replace('-', '_')) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from dash import Dash, html, dcc, Input, Output

## Paso 0 · Generar predicciones (ya está hecho: solo ejecuta)

In [ ]:
RUTA = "predicciones_digits.csv"
if not os.path.exists(RUTA):
    X, y = load_digits(return_X_y=True)
    modelos = {"Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
               "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
               "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    filas = []
    for nombre, m in modelos.items():
        P = cross_val_predict(m, X, y, cv=cv, method="predict_proba")
        filas.append(pd.DataFrame({"id": np.arange(len(y)), "modelo": nombre, "y_true": y,
                                   "y_pred": P.argmax(axis=1), "confianza": P.max(axis=1).round(4)}))
    pd.concat(filas).to_csv(RUTA, index=False)
print("CSV listo:", RUTA)

In [ ]:
# Estilos reutilizables (CSS escrito como diccionarios de Python)
COLOR = {"fondo": "#F5F7FA", "texto": "#1F2937", "primario": "#1F4E79", "suave": "#6B7280"}
ESTILO_PAGINA = {"fontFamily": "Arial, sans-serif", "backgroundColor": COLOR["fondo"],
                 "padding": "20px", "color": COLOR["texto"]}
ESTILO_TARJETA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "14px 18px",
                  "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "160px",
                  "boxSizing": "border-box"}
ESTILO_FILA = {"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "16px"}
ESTILO_CAJA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "8px",
               "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "380px"}

def tarjeta(titulo, valor, id_valor=None):
    """Tarjeta KPI: título pequeño + valor grande. Con id_valor, el valor se puede actualizar con un callback."""
    props = {"id": id_valor} if id_valor else {}
    return html.Div([
        html.Div(titulo, style={"fontSize": "13px", "color": COLOR["suave"]}),
        html.Div(valor, style={"fontSize": "28px", "fontWeight": "700", "color": COLOR["primario"]}, **props),
    ], style=ESTILO_TARJETA)

## Paso 1 · Cargar el CSV
Columnas: `modelo`, `y_true` (dígito real), `y_pred` (dígito predicho) y `confianza` (qué tan seguro estaba el modelo).

In [ ]:
pred = ...              # TODO: pd.read_csv(RUTA)
pred["acierto"] = ...   # TODO: True si y_true == y_pred
MODELOS = sorted(pred["modelo"].unique())
print(pred.shape, MODELOS)
pred.head()

## Paso 2 · Accuracy de cada modelo
La **accuracy** es el % de predicciones correctas. Calcúlala para cada modelo y grafícala en barras.

In [ ]:
acc = ...  # TODO: promedio de "acierto" por modelo (groupby), ordenado, × 100
display(acc.round(1))
fig = ...  # TODO: px.bar horizontal (orientation="h") con text_auto=".1f" y range_x=[90, 100]
fig

In [ ]:
# ✅ verificación
assert acc.between(90, 100).all()
print("Paso 2 ✔")

## Paso 3 · Matriz de confusión
Filas = dígito real, columnas = dígito predicho. La diagonal son los aciertos; todo lo demás son errores.

In [ ]:
def fig_confusion(df):
    cm = ...   # TODO: confusion_matrix(df["y_true"], df["y_pred"], labels=range(10))
    fig = ...  # TODO: px.imshow(cm, text_auto=True, color_continuous_scale="Blues", ...)
    return fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

fig_confusion(pred[pred.modelo == "KNN (k=5)"])

**Pregunta:** ¿qué dígito confunde más el modelo y con cuál? ¿Tiene sentido visualmente?

*Tu respuesta:* …

## Paso 4 · % de aciertos por dígito
Para un modelo, calcula el % de aciertos de cada dígito real y grafícalo en barras.

In [ ]:
def fig_aciertos_digito(df):
    por_digito = ...  # TODO: (df.groupby("y_true")["acierto"].mean() * 100).reset_index()
    fig = ...         # TODO: px.bar con x="y_true", y="acierto", text_auto=".0f", range_y=[80, 101]
    return fig.update_layout(xaxis=dict(dtick=1), margin=dict(t=50, l=10, r=10, b=10))

fig_aciertos_digito(pred[pred.modelo == "KNN (k=5)"])

## Paso 5 · Layout + callback
Un `dcc.Dropdown` para elegir el modelo (id `f-modelo`), dos tarjetas (accuracy `k-acc` y número de errores `k-err`) y dos gráficos (matriz `g-conf` y aciertos por dígito `g-dig`).

In [ ]:
app = Dash(__name__)
app.layout = html.Div([
    html.H1("Lectura automática de dígitos · ¿Qué modelo usar?"),
    html.Div([html.Label("Modelo"), ...],     # TODO: dcc.Dropdown(id="f-modelo", options=MODELOS, value=MODELOS[0], clearable=False)
             style={**ESTILO_TARJETA, "marginBottom": "16px", "maxWidth": "420px"}),
    html.Div([...], style=ESTILO_FILA),       # TODO: tarjeta("Accuracy", "", "k-acc") y tarjeta("Errores", "", "k-err")
    html.Div([...], style=ESTILO_FILA),       # TODO: 2 html.Div(dcc.Graph(id=...), style=ESTILO_CAJA): g-conf y g-dig
], style=ESTILO_PAGINA)

@app.callback(
    # TODO: 2 Output de tarjetas ("children") y 2 de gráficos ("figure")
    # TODO: 1 Input: f-modelo ("value")
)
def actualizar(modelo):
    df = ...    # TODO: filas del modelo elegido
    return ...  # TODO: accuracy en texto, errores en texto, fig_confusion(df), fig_aciertos_digito(df)

print(actualizar(MODELOS[0])[:2])  # prueba sin servidor

In [ ]:
app.run(jupyter_mode="inline", port=8081, height=900)

## Paso 6 · Conclusiones
Explora el tablero y completa con números concretos.

1. **Mejor modelo:** …
2. **Dígitos donde más se equivoca:** …
3. **Recomendación:** …

### ⭐ Retos
- Agrega una tarjeta con la **confianza promedio** del modelo (columna `confianza`).
- Agrega un `dcc.Slider` de confianza mínima: muestra solo las predicciones con `confianza ≥` el valor elegido. ¿Qué pasa con la accuracy y con cuántos casos quedan?